# Python Data Pipeline Engineering — Colab

Notebook สำหรับ Lab: **Incremental + Idempotent ETL Pipeline**  
รองรับ `batch_1 → batch_1 ซ้ำ → batch_2 → batch_3`

> อัปโหลดไฟล์ Dataset `.xlsx` ใน Cell Upload แล้วกด **Runtime → Run all**


In [ ]:
# 1) Install / imports
!pip -q install openpyxl

import os
import re
import sqlite3
import logging
from dataclasses import dataclass
from datetime import datetime

import pandas as pd
import numpy as np
from google.colab import files
from IPython.display import display

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

print("Ready")


## 2) Upload Dataset

อัปโหลดไฟล์ `Python_Data_Pipeline_Lab_Dataset (1).xlsx`


In [ ]:
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("ไม่พบไฟล์ Dataset")

INPUT_XLSX = next(iter(uploaded.keys()))
print("Using:", INPUT_XLSX)

xls = pd.ExcelFile(INPUT_XLSX)
print("Sheets:", xls.sheet_names)


In [ ]:
# 3) Read workbook and inspect sheets
sheets = {}

for sheet in xls.sheet_names:
    df = pd.read_excel(INPUT_XLSX, sheet_name=sheet)
    sheets[sheet] = df
    print(f"\n--- {sheet} ---")
    print("shape:", df.shape)
    display(df.head())



In [ ]:
# 4) Identify customer/product/order sheets automatically
def norm_name(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower())

sheet_map = {norm_name(k): k for k in sheets}

def find_sheet(patterns):
    for key, original in sheet_map.items():
        if any(p in key for p in patterns):
            return original
    return None

customer_sheet = find_sheet(["customer"])
product_sheet = find_sheet(["product"])
order_sheets = [orig for key, orig in sheet_map.items() if "order" in key or "batch" in key]

print("customer_sheet =", customer_sheet)
print("product_sheet  =", product_sheet)
print("order_sheets   =", order_sheets)

if customer_sheet is None or product_sheet is None:
    raise ValueError("หา sheet customers/products ไม่พบจากไฟล์ Dataset")

customers = sheets[customer_sheet].copy()
products = sheets[product_sheet].copy()

print("Customers:", customers.shape)
print("Products :", products.shape)


In [ ]:
# 5) Normalize column names
def clean_columns(df):
    df = df.copy()
    df.columns = [
        re.sub(r"[^a-z0-9_]+", "_", str(c).strip().lower()).strip("_")
        for c in df.columns
    ]
    return df

customers = clean_columns(customers)
products = clean_columns(products)
for k in list(sheets):
    sheets[k] = clean_columns(sheets[k])

print("Customer columns:", customers.columns.tolist())
print("Product columns :", products.columns.tolist())


In [ ]:
# 6) Find required columns robustly
def pick_col(df, candidates, required=True):
    cols = list(df.columns)
    for cand in candidates:
        c = norm_name(cand)
        for col in cols:
            if norm_name(col) == c or c in norm_name(col):
                return col
    if required:
        raise KeyError(f"ไม่พบ column ใด ๆ จาก {candidates}. Available: {cols}")
    return None

# Dimension columns
cust_id = pick_col(customers, ["customer_id"])
cust_name = pick_col(customers, ["customer_name", "name"], required=False)
cust_province = pick_col(customers, ["province"], required=False)
cust_segment = pick_col(customers, ["segment"], required=False)

prod_id = pick_col(products, ["product_id"])
prod_name = pick_col(products, ["product_name", "name"], required=False)
prod_category = pick_col(products, ["category"], required=False)

def ensure_col(df, col, default=""):
    if col is None:
        return pd.Series([default] * len(df), index=df.index)
    return df[col]

customers_std = pd.DataFrame({
    "customer_id": customers[cust_id].astype(str).str.strip(),
    "customer_name": ensure_col(customers, cust_name).astype(str).str.strip(),
    "province": ensure_col(customers, cust_province).astype(str).str.strip(),
    "segment": ensure_col(customers, cust_segment).astype(str).str.strip()
})

products_std = pd.DataFrame({
    "product_id": products[prod_id].astype(str).str.strip(),
    "product_name": ensure_col(products, prod_name).astype(str).str.strip(),
    "category": ensure_col(products, prod_category).astype(str).str.strip()
})

customers = customers_std.drop_duplicates("customer_id").reset_index(drop=True)
products = products_std.drop_duplicates("product_id").reset_index(drop=True)

print("Standardized dimensions:")
display(customers.head())
display(products.head())


In [ ]:
# 7) Pipeline configuration
@dataclass
class PipelineConfig:
    input_path: str
    output_db: str
    batch_list: list
    error_mode: str = "quarantine"

config = PipelineConfig(
    input_path=INPUT_XLSX,
    output_db="/content/retail_dw.db",
    batch_list=[1, 2, 3],
    error_mode="quarantine"
)

DB_PATH = config.output_db
print(config)


In [ ]:
# 8) Database initialization
def connect_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

def init_db():
    conn = connect_db()
    conn.executescript("""
    CREATE TABLE IF NOT EXISTS dim_customer (
        customer_key INTEGER PRIMARY KEY AUTOINCREMENT,
        customer_id TEXT UNIQUE NOT NULL,
        customer_name TEXT,
        province TEXT,
        segment TEXT
    );

    CREATE TABLE IF NOT EXISTS dim_product (
        product_key INTEGER PRIMARY KEY AUTOINCREMENT,
        product_id TEXT UNIQUE NOT NULL,
        product_name TEXT,
        category TEXT
    );

    CREATE TABLE IF NOT EXISTS dim_date (
        date_key INTEGER PRIMARY KEY,
        full_date TEXT UNIQUE,
        day INTEGER,
        month INTEGER,
        quarter INTEGER,
        year INTEGER
    );

    CREATE TABLE IF NOT EXISTS fact_sales (
        order_id TEXT PRIMARY KEY,
        date_key INTEGER NOT NULL,
        customer_key INTEGER NOT NULL,
        product_key INTEGER NOT NULL,
        quantity REAL NOT NULL,
        unit_price REAL NOT NULL,
        discount_pct REAL NOT NULL,
        gross_amount REAL NOT NULL,
        net_amount REAL NOT NULL,
        payment_method TEXT,
        sales_channel TEXT,
        updated_at TEXT,
        source_batch INTEGER,
        FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
        FOREIGN KEY (customer_key) REFERENCES dim_customer(customer_key),
        FOREIGN KEY (product_key) REFERENCES dim_product(product_key)
    );

    CREATE TABLE IF NOT EXISTS quarantine (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        order_id TEXT,
        reason_code TEXT NOT NULL,
        source_batch INTEGER,
        raw_data TEXT,
        created_at TEXT
    );

    CREATE TABLE IF NOT EXISTS pipeline_run_log (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        batch INTEGER,
        started_at TEXT,
        ended_at TEXT,
        rows_read INTEGER,
        rows_valid INTEGER,
        rows_rejected INTEGER,
        rows_duplicated INTEGER,
        rows_loaded INTEGER,
        status TEXT
    );
    """)
    conn.commit()
    conn.close()
    print("Database initialized:", DB_PATH)

# IMPORTANT: This is called BEFORE load_dimensions()
init_db()


In [ ]:
# 9) Load dimensions
def load_dimensions():
    init_db()
    conn = connect_db()

    conn.executemany(
        """INSERT OR IGNORE INTO dim_customer
        (customer_id, customer_name, province, segment)
        VALUES (?, ?, ?, ?)""",
        customers[["customer_id","customer_name","province","segment"]]
        .itertuples(index=False, name=None)
    )

    conn.executemany(
        """INSERT OR IGNORE INTO dim_product
        (product_id, product_name, category)
        VALUES (?, ?, ?)""",
        products[["product_id","product_name","category"]]
        .itertuples(index=False, name=None)
    )

    conn.commit()
    conn.close()
    print("Dimensions loaded:", len(customers), "customers,", len(products), "products")

load_dimensions()


In [ ]:
# 10) Locate batch dataframes
batch_dfs = {}

for original in order_sheets:
    df = sheets[original].copy()
    n = norm_name(original)

    m = re.search(r"(?:batch|order).*?([123])", n)
    if m:
        batch_no = int(m.group(1))
    else:
        # fallback: infer from order sheet order
        continue

    batch_dfs[batch_no] = df

# If sheet names don't contain a detectable batch number, use order sheets in order.
if set(batch_dfs.keys()) != {1,2,3} and len(order_sheets) >= 3:
    batch_dfs = {i+1: sheets[order_sheets[i]].copy() for i in range(3)}

print({k: v.shape for k,v in batch_dfs.items()})

if set(batch_dfs.keys()) != {1,2,3}:
    raise ValueError("ต้องมีข้อมูล order batch 1, 2 และ 3 ใน Dataset")


In [ ]:
# 11) Standardize order columns
ALIASES = {
    "order_id": ["order_id", "orderid"],
    "customer_id": ["customer_id", "customerid"],
    "product_id": ["product_id", "productid"],
    "order_date": ["order_date", "date", "orderdate"],
    "quantity": ["quantity", "qty"],
    "unit_price": ["unit_price", "price", "unitprice"],
    "discount_pct": ["discount_pct", "discount", "discountpercent", "discount_percent"],
    "payment_method": ["payment_method", "payment", "paymentmethod"],
    "sales_channel": ["sales_channel", "channel", "saleschannel"],
    "updated_at": ["updated_at", "updatedat", "last_updated"]
}

def standardize_orders(df):
    df = clean_columns(df.copy())
    out = pd.DataFrame(index=df.index)

    for target, candidates in ALIASES.items():
        col = pick_col(df, candidates, required=False)
        out[target] = df[col] if col is not None else np.nan

    return out

orders = {b: standardize_orders(df) for b, df in batch_dfs.items()}

for b, df in orders.items():
    print(f"Batch {b}:")
    print(df.columns.tolist())
    display(df.head(3))


In [ ]:
# 12) Transform + validation helpers
PAYMENT_MAP = {
    "cash": "cash",
    "เงินสด": "cash",
    "credit": "credit_card",
    "creditcard": "credit_card",
    "credit_card": "credit_card",
    "card": "credit_card",
    "โอน": "bank_transfer",
    "transfer": "bank_transfer",
    "bank_transfer": "bank_transfer",
    "promptpay": "promptpay",
    "online": "online"
}

CHANNEL_MAP = {
    "store": "store",
    "หน้าร้าน": "store",
    "online": "online",
    "เว็บ": "online",
    "website": "online",
    "app": "app",
    "marketplace": "marketplace",
    "โทรศัพท์": "phone",
    "phone": "phone"
}

def normalize_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

def transform_validate(df, batch_no):
    d = df.copy()

    # Safe conversion
    d["order_id"] = d["order_id"].astype("string").str.strip()
    d["customer_id"] = d["customer_id"].astype("string").str.strip()
    d["product_id"] = d["product_id"].astype("string").str.strip()

    d["order_date"] = pd.to_datetime(d["order_date"], errors="coerce")
    d["updated_at"] = pd.to_datetime(d["updated_at"], errors="coerce")

    for col in ["quantity", "unit_price", "discount_pct"]:
        d[col] = pd.to_numeric(d[col], errors="coerce")

    d["payment_method"] = d["payment_method"].map(
        lambda x: PAYMENT_MAP.get(normalize_text(x), normalize_text(x))
    )
    d["sales_channel"] = d["sales_channel"].map(
        lambda x: CHANNEL_MAP.get(normalize_text(x), normalize_text(x))
    )

    cust_set = set(customers["customer_id"].astype(str))
    prod_set = set(products["product_id"].astype(str))

    reasons = []

    for _, r in d.iterrows():
        rs = []

        if pd.isna(r["order_id"]) or str(r["order_id"]).strip() == "":
            rs.append("MISSING_ORDER_ID")

        if pd.isna(r["order_date"]):
            rs.append("INVALID_DATE")

        if pd.isna(r["quantity"]) or r["quantity"] <= 0:
            rs.append("INVALID_QUANTITY")

        if pd.isna(r["unit_price"]) or r["unit_price"] <= 0:
            rs.append("INVALID_UNIT_PRICE")

        if pd.isna(r["discount_pct"]) or not (0 <= r["discount_pct"] <= 100):
            rs.append("INVALID_DISCOUNT")

        if str(r["customer_id"]) not in cust_set:
            rs.append("INVALID_CUSTOMER_ID")

        if str(r["product_id"]) not in prod_set:
            rs.append("INVALID_PRODUCT_ID")

        reasons.append("|".join(rs))

    d["reason_code"] = reasons

    clean = d[d["reason_code"] == ""].copy()
    rejected = d[d["reason_code"] != ""].copy()

    # Deduplicate: latest updated_at per order_id
    duplicated_count = 0
    if len(clean):
        clean = clean.sort_values("updated_at", na_position="first")
        before = len(clean)
        clean = clean.drop_duplicates("order_id", keep="last")
        duplicated_count = before - len(clean)

    if len(clean):
        clean["gross_amount"] = clean["quantity"] * clean["unit_price"]
        clean["net_amount"] = clean["gross_amount"] * (
            1 - clean["discount_pct"] / 100
        )

    return clean, rejected, duplicated_count


In [ ]:
# 13) Date dimension
def ensure_dates(conn, dates):
    for dt in pd.to_datetime(pd.Series(dates), errors="coerce").dropna().dt.normalize().unique():
        ts = pd.Timestamp(dt)
        date_key = int(ts.strftime("%Y%m%d"))
        conn.execute(
            """INSERT OR IGNORE INTO dim_date
            (date_key, full_date, day, month, quarter, year)
            VALUES (?, ?, ?, ?, ?, ?)""",
            (
                date_key,
                ts.strftime("%Y-%m-%d"),
                int(ts.day),
                int(ts.month),
                int(ts.quarter),
                int(ts.year)
            )
        )


In [ ]:
# 14) Extract + Transform + Load
def run_pipeline(config, batch_no):
    started = datetime.now().isoformat(timespec="seconds")

    if batch_no not in orders:
        raise ValueError(f"ไม่พบ batch {batch_no}")

    raw = orders[batch_no].copy()
    rows_read = len(raw)

    try:
        clean, rejected, duplicated = transform_validate(raw, batch_no)

        conn = connect_db()

        # Quarantine bad rows
        for _, r in rejected.iterrows():
            raw_data = r.to_json(date_format="iso")
            conn.execute(
                """INSERT INTO quarantine
                (order_id, reason_code, source_batch, raw_data, created_at)
                VALUES (?, ?, ?, ?, ?)""",
                (
                    None if pd.isna(r["order_id"]) else str(r["order_id"]),
                    r["reason_code"],
                    batch_no,
                    raw_data,
                    datetime.now().isoformat(timespec="seconds")
                )
            )

        # Load dates and fact
        if len(clean):
            ensure_dates(conn, clean["order_date"])

        rows_loaded = 0

        for _, r in clean.iterrows():
            date_key = int(pd.Timestamp(r["order_date"]).strftime("%Y%m%d"))

            cust = conn.execute(
                "SELECT customer_key FROM dim_customer WHERE customer_id=?",
                (str(r["customer_id"]),)
            ).fetchone()

            prod = conn.execute(
                "SELECT product_key FROM dim_product WHERE product_id=?",
                (str(r["product_id"]),)
            ).fetchone()

            if not cust or not prod:
                continue

            # Idempotent: same order_id is updated only when source updated_at is newer
            existing = conn.execute(
                "SELECT updated_at FROM fact_sales WHERE order_id=?",
                (str(r["order_id"]),)
            ).fetchone()

            should_write = False

            if existing is None:
                should_write = True
            else:
                old = pd.to_datetime(existing[0], errors="coerce")
                new = pd.to_datetime(r["updated_at"], errors="coerce")

                if pd.isna(old):
                    should_write = True
                elif not pd.isna(new) and new > old:
                    should_write = True

            if should_write:
                conn.execute(
                    """INSERT INTO fact_sales
                    (order_id, date_key, customer_key, product_key,
                     quantity, unit_price, discount_pct,
                     gross_amount, net_amount,
                     payment_method, sales_channel, updated_at, source_batch)
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                    ON CONFLICT(order_id) DO UPDATE SET
                        date_key=excluded.date_key,
                        customer_key=excluded.customer_key,
                        product_key=excluded.product_key,
                        quantity=excluded.quantity,
                        unit_price=excluded.unit_price,
                        discount_pct=excluded.discount_pct,
                        gross_amount=excluded.gross_amount,
                        net_amount=excluded.net_amount,
                        payment_method=excluded.payment_method,
                        sales_channel=excluded.sales_channel,
                        updated_at=excluded.updated_at,
                        source_batch=excluded.source_batch
                    """,
                    (
                        str(r["order_id"]),
                        date_key,
                        cust[0],
                        prod[0],
                        float(r["quantity"]),
                        float(r["unit_price"]),
                        float(r["discount_pct"]),
                        float(r["gross_amount"]),
                        float(r["net_amount"]),
                        r["payment_method"],
                        r["sales_channel"],
                        None if pd.isna(r["updated_at"]) else pd.Timestamp(r["updated_at"]).isoformat(),
                        batch_no
                    )
                )
                rows_loaded += 1

        ended = datetime.now().isoformat(timespec="seconds")

        conn.execute(
            """INSERT INTO pipeline_run_log
            (batch, started_at, ended_at, rows_read, rows_valid,
             rows_rejected, rows_duplicated, rows_loaded, status)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""",
            (
                batch_no, started, ended, rows_read, len(clean),
                len(rejected), duplicated, rows_loaded, "SUCCESS"
            )
        )

        conn.commit()
        conn.close()

        return {
            "batch": batch_no,
            "rows_read": rows_read,
            "valid": len(clean),
            "rejected": len(rejected),
            "duplicated": duplicated,
            "loaded": rows_loaded,
            "status": "SUCCESS"
        }

    except Exception as e:
        ended = datetime.now().isoformat(timespec="seconds")

        try:
            conn = connect_db()
            conn.execute(
                """INSERT INTO pipeline_run_log
                (batch, started_at, ended_at, rows_read, rows_valid,
                 rows_rejected, rows_duplicated, rows_loaded, status)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""",
                (batch_no, started, ended, rows_read, 0, 0, 0, 0, f"FAILED: {e}")
            )
            conn.commit()
            conn.close()
        except Exception:
            pass

        raise


## 15) Acceptance Test

รัน 4 รอบตามโจทย์:

1. `batch_1`
2. `batch_1` ซ้ำ
3. `batch_2`
4. `batch_3`

**ก่อนเริ่มจะ reset database และสร้างตารางใหม่ทุกครั้ง** ดังนั้นจะไม่เกิด `no such table: dim_customer`


In [ ]:
# 15) RESET DB + CREATE TABLES + ACCEPTANCE TEST

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

# สำคัญ: ต้องสร้าง tables ก่อน load_dimensions()
init_db()
load_dimensions()

results = []

for b in [1, 1, 2, 3]:
    result = run_pipeline(config, b)
    results.append(result)
    print(result)

results_df = pd.DataFrame(results)
display(results_df)


In [ ]:
# 16) Verify database tables and fact count
conn = connect_db()

tables = pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
""", conn)

fact_count = pd.read_sql_query(
    "SELECT COUNT(*) AS fact_rows FROM fact_sales", conn
)

duplicate_orders = pd.read_sql_query("""
SELECT order_id, COUNT(*) AS n
FROM fact_sales
GROUP BY order_id
HAVING COUNT(*) > 1
""", conn)

display(tables)
display(fact_count)

print("Duplicate order_id count:", len(duplicate_orders))

conn.close()


In [ ]:
# 17) KPI summary
conn = connect_db()

kpi = pd.read_sql_query("""
SELECT
    SUM(rows_read) AS rows_read_total,
    SUM(rows_valid) AS valid_total,
    SUM(rows_rejected) AS rejected_total,
    SUM(rows_duplicated) AS duplicated_total,
    SUM(rows_loaded) AS loaded_total
FROM pipeline_run_log
""", conn)

sales = pd.read_sql_query("""
SELECT
    COUNT(*) AS fact_rows,
    COALESCE(SUM(net_amount), 0) AS total_net_sales
FROM fact_sales
""", conn)

display(kpi)
display(sales)

conn.close()


In [ ]:
# 18) Export required deliverables
conn = connect_db()

pd.read_sql_query(
    "SELECT * FROM quarantine ORDER BY id",
    conn
).to_csv("/content/quarantine.csv", index=False)

pd.read_sql_query(
    "SELECT * FROM pipeline_run_log ORDER BY id",
    conn
).to_csv("/content/pipeline_run_log.csv", index=False)

readme = """# Retail Data Warehouse ETL

## Run
Open the notebook in Google Colab, upload the provided XLSX dataset, then Run all.

## Pipeline
Extract -> Transform -> Validate -> Quarantine -> Load

## Star Schema
- dim_customer
- dim_product
- dim_date
- fact_sales

## Idempotency
fact_sales uses order_id as the primary key. Re-running an existing batch does not create duplicate fact rows. A record is updated only when its updated_at is newer.

## Outputs
- retail_dw.db
- quarantine.csv
- pipeline_run_log.csv
"""

with open("/content/README.md", "w", encoding="utf-8") as f:
    f.write(readme)

conn.close()

print("Files created:")
for p in [DB_PATH, "/content/quarantine.csv", "/content/pipeline_run_log.csv", "/content/README.md"]:
    print(p, os.path.getsize(p) if os.path.exists(p) else "NOT FOUND")


## 19) Download ผลลัพธ์

Cell นี้จะดาวน์โหลดไฟล์ที่ต้องส่งตามโจทย์


In [ ]:
# 19) Download deliverables
from google.colab import files

for path in [
    "/content/retail_dw.db",
    "/content/quarantine.csv",
    "/content/pipeline_run_log.csv",
    "/content/README.md"
]:
    if os.path.exists(path):
        files.download(path)
